# Camada Silver — `ecommerce_clientes` 

Lê a Bronze física em `az://squad1/bronze/ecommerce_clientes`, aplica as 10 regras de qualidade, grava **somente linhas válidas** na Silver física em `az://squad1/silver/ecommerce_clientes` e registra as falhas em `az://squad1/dq_monitoring_logs`.

Este notebook possui modo de reprocessamento para quando os arquivos já foram lidos anteriormente.

In [0]:
%run ../utils/utils.ipynb

##  Imports e parâmetros

In [0]:


#  Carrega as funções utilitárias (gravar_delta, ler_delta, etc)


import uuid
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime, timezone
from functools import reduce

# Variáveis do Processo
RUN_ID = str(uuid.uuid4())
TABELA_ALVO = "ecommerce_clientes"
TABELA_DQ = "dq_monitoring_logs"
DATA_EXECUCAO = datetime.now(timezone.utc)

print(f"Iniciando processamento Silver - Clientes - Run ID: {RUN_ID}")

## Leitura do Micro-lote e Tabelas de Referência (Joins)

In [0]:
# 1. Carrega a tabela Bronze de Clientes
try:
    df_bronze_clientes = ler_delta("bronze", TABELA_ALVO, STORAGE_OPTIONS)
except Exception as e:
    raise Exception(f"Erro: A tabela Bronze de {TABELA_ALVO} não foi encontrada.")

# 2. Isola o Micro-lote (Apenas clientes que ainda não estão na Silver)
if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_silver_atual = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    df_micro_lote = df_bronze_clientes.join(df_silver_atual, "id_cliente", "left_anti")
else:
    df_micro_lote = df_bronze_clientes

qtd_novos = df_micro_lote.count()
print(f"Registros novos no micro-lote de clientes: {qtd_novos}")

# =================================================================================
# 3. Leitura das Tabelas de Referência para validação de Chaves (Regras 7 e 8)
# =================================================================================
def obter_referencia_ids(camada, tabela, coluna_id):
    if delta_existe(camada, tabela, STORAGE_OPTIONS):
        return ler_delta(camada, tabela, STORAGE_OPTIONS).select(coluna_id).dropDuplicates()
    else:
        schema = StructType([StructField(coluna_id, LongType(), True)])
        return spark.createDataFrame([], schema)

# Carrega os IDs de quem já tem endereço e quem já fez pedido
df_enderecos_ref = obter_referencia_ids("bronze", "ecommerce_enderecos", "id_cliente") \
    .withColumnRenamed("id_cliente", "id_cliente_tem_endereco")

df_pedidos_ref = obter_referencia_ids("bronze", "ecommerce_pedidos", "id_cliente") \
    .withColumnRenamed("id_cliente", "id_cliente_tem_pedido")

print("Tabelas de referência para validação cruzada carregadas.")

## Leitura da Bronze e Aplicação das 10 Regras de Data Quality



In [0]:
if qtd_novos > 0:
    # Expressões regulares e listas para validação
    regex_email = r"^.+@.+\..+$"
    regex_uuid = r"^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$"
    provedores_validos = ["gmail", "yahoo", "hotmail", "outlook", "uol", "bol", "terra"]
    
    # Janelas para checar duplicidade no próprio lote
    w_id_cliente = Window.partitionBy("id_cliente")
    w_email = Window.partitionBy("email")
    w_uuid = Window.partitionBy("uuid_cliente")

    # Prepara o DataFrame base aplicando as transformações de tipo e os Joins cruzados
    df_base = df_micro_lote \
        .withColumn("dt_cadastro_ts", F.col("dt_cadastro").cast("timestamp")) \
        .withColumn("dt_atualizacao_ts", F.col("dt_ultima_atualizacao").cast("timestamp")) \
        .withColumn("qtd_id_cliente", F.count("*").over(w_id_cliente)) \
        .withColumn("qtd_email", F.count("*").over(w_email)) \
        .withColumn("qtd_uuid", F.count("*").over(w_uuid)) \
        .join(df_enderecos_ref, df_micro_lote.id_cliente == df_enderecos_ref.id_cliente_tem_endereco, "left_outer") \
        .join(df_pedidos_ref, df_micro_lote.id_cliente == df_pedidos_ref.id_cliente_tem_pedido, "left_outer")

    # Extração do provedor de e-mail (tudo após o @ e antes do primeiro ponto)
    # Note o 'r' minúsculo antes de "\." Usado para evitar Warning no código
    df_base = df_base.withColumn("provedor", F.split(F.split(F.col("email"), "@")[1], r"\.")[0])
    # Aplicação massiva das 10 regras de qualidade
    df_silver_clientes = df_base \
        .withColumn("r1_id_cliente_falhou", F.col("id_cliente").isNull() | (F.col("id_cliente").cast("string") == "") | (F.col("qtd_id_cliente") > 1)) \
        .withColumn("r2_email_falhou", F.col("email").isNull() | (~F.col("email").rlike(regex_email)) | (F.col("qtd_email") > 1)) \
        .withColumn("r3_nome_sobrenome_falhou", F.col("nome").isNull() | (F.trim(F.col("nome")) == "") | F.col("sobrenome").isNull() | (F.trim(F.col("sobrenome")) == "")) \
        .withColumn("r4_senha_hash_falhou", F.col("senha_hash").isNull() | (F.length(F.col("senha_hash")) != 64)) \
        .withColumn("r5_dt_cadastro_falhou", F.col("dt_cadastro_ts").isNull() | (F.col("dt_cadastro_ts") > F.current_timestamp())) \
        .withColumn("r6_dt_atualizacao_falhou", F.col("dt_cadastro_ts").isNull() | F.col("dt_atualizacao_ts").isNull() | (F.col("dt_atualizacao_ts") < F.col("dt_cadastro_ts"))) \
        .withColumn("r7_endereco_associado_falhou", F.col("id_cliente_tem_endereco").isNull()) \
        .withColumn("r8_pedido_associado_falhou", F.col("id_cliente_tem_pedido").isNull()) \
        .withColumn("r9_provedor_reconhecido_falhou", F.col("email").isNotNull() & (~F.col("provedor").isin(provedores_validos))) \
        .withColumn("r10_uuid_cliente_falhou", F.col("uuid_cliente").isNull() | (~F.col("uuid_cliente").rlike(regex_uuid)) | (F.col("qtd_uuid") > 1))

    # Separação de Severidade (Regras Críticas vs Avisos)
    regras_criticas = [
        "r1_id_cliente_falhou", "r2_email_falhou", "r3_nome_sobrenome_falhou", 
        "r4_senha_hash_falhou", "r5_dt_cadastro_falhou", "r6_dt_atualizacao_falhou", 
        "r10_uuid_cliente_falhou"
    ]
    
    condicao_falha_critica = F.expr(" OR ".join(regras_criticas))

    df_silver_clientes = df_silver_clientes \
        .withColumn("silver_linha_valida", ~condicao_falha_critica) \
        .withColumn("silver_processed_at", F.current_timestamp()) \
        .withColumn("silver_run_id", F.lit(RUN_ID))
    
    print("Muralha de qualidade de clientes estruturada com sucesso.")
else:
    print("Nenhum cliente novo encontrado para processamento.")

## Geração do Log Unificado (DQ Monitoring)

In [0]:
if qtd_novos > 0:
    catalogo_regras = [
        {"coluna": "r1_id_cliente_falhou", "regra": "R1_ID_CLIENTE_NULO_DUPLICADO", "severidade": "Critica"},
        {"coluna": "r2_email_falhou", "regra": "R2_EMAIL_INVALIDO_DUPLICADO", "severidade": "Critica"},
        {"coluna": "r3_nome_sobrenome_falhou", "regra": "R3_NOME_SOBRENOME_VAZIO", "severidade": "Critica"},
        {"coluna": "r4_senha_hash_falhou", "regra": "R4_SENHA_HASH_TAMANHO_ERRADO", "severidade": "Critica"},
        {"coluna": "r5_dt_cadastro_falhou", "regra": "R5_DATA_CADASTRO_NULA_FUTURA", "severidade": "Critica"},
        {"coluna": "r6_dt_atualizacao_falhou", "regra": "R6_DATA_ATUALIZACAO_ANTERIOR_CADASTRO", "severidade": "Critica"},
        {"coluna": "r7_endereco_associado_falhou", "regra": "R7_CLIENTE_SEM_ENDERECO", "severidade": "Aviso"},
        {"coluna": "r8_pedido_associado_falhou", "regra": "R8_CLIENTE_INATIVO_SEM_PEDIDO", "severidade": "Aviso"},
        {"coluna": "r9_provedor_reconhecido_falhou", "regra": "R9_PROVEDOR_EMAIL_NAO_RECONHECIDO", "severidade": "Aviso"},
        {"coluna": "r10_uuid_cliente_falhou", "regra": "R10_UUID_INVALIDO_DUPLICADO", "severidade": "Critica"}
    ]

    total_registros = df_silver_clientes.count()
    logs_list = []
    
    for r in catalogo_regras:
        qtd_falhas = df_silver_clientes.filter(F.col(r["coluna"]) == True).count()
        if qtd_falhas > 0:
            logs_list.append((
                RUN_ID,
                TABELA_ALVO,
                r["regra"],
                "FAIL",
                r["severidade"],
                int(qtd_falhas),
                int(total_registros),
                datetime.now(timezone.utc),
                f"Bronze Delta ({TABELA_ALVO})"
            ))

    # Cria o DataFrame usando a nossa função salva no utils.ipynb
    if logs_list:
        df_dq_monitoring_logs_novos = spark.createDataFrame(logs_list, schema_dq_logs())
    else:
        df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())

    print("Logs gerados prontos para gravação:", df_dq_monitoring_logs_novos.count())
    display(df_dq_monitoring_logs_novos)
else:
    df_dq_monitoring_logs_novos = spark.createDataFrame([], schema_dq_logs())
    print("Sem novos logs: lote vazio.")

## Gravação Final (Silver Clientes e Logs)

In [0]:
if qtd_novos > 0:
    # 1. Isola apenas as colunas originais do cliente para salvar na Silver limpa
    colunas_finais = df_micro_lote.columns + ["silver_processed_at", "silver_run_id"]
    
    df_silver_validos = df_silver_clientes \
        .filter(F.col("silver_linha_valida") == True) \
        .select(*colunas_finais)
        
    qtd_validos = df_silver_validos.count()
    print(f"Clientes aprovados para a Silver: {qtd_validos}")

    # 2. Gravação na Silver (Dimensional: Sem partição física)
    if qtd_validos > 0:
        sucesso_silver = gravar_delta(
            df=df_silver_validos,
            camada="silver",
            tabela=TABELA_ALVO,
            storage_opts=STORAGE_OPTIONS,
            mode="append",
            particionar=False 
        )
        if sucesso_silver:
            print("Tabela Silver de Clientes atualizada com sucesso!")

    # 3. Gravação dos Logs de Data Quality (Com controle preventivo de duplicidade)
    if df_dq_monitoring_logs_novos.count() > 0:
        
        # Verifica se já existem logs gravados hoje para essa mesma tabela
        if delta_existe("silver", TABELA_DQ, STORAGE_OPTIONS):
            df_logs_historico = ler_delta("silver", TABELA_DQ, STORAGE_OPTIONS) \
                .filter(F.col("tabela") == TABELA_ALVO)
            
            # Condição: Mesma Regra e Mesma Data (ignorando horas)
            condicao_join = [
                df_dq_monitoring_logs_novos.regra == df_logs_historico.regra,
                F.to_date(df_dq_monitoring_logs_novos.timestamp_execucao) == F.to_date(df_logs_historico.timestamp_execucao)
            ]
            
            # Anti-Join: Mantém apenas os logs que NÃO estão no histórico de hoje
            df_logs_para_gravar = df_dq_monitoring_logs_novos.join(df_logs_historico, condicao_join, "left_anti")
            print(f"Filtro aplicado: {df_logs_para_gravar.count()} logs inéditos liberados para gravação.")
        else:
            df_logs_para_gravar = df_dq_monitoring_logs_novos
            
        # Grava apenas se sobrou algo após o filtro
        if df_logs_para_gravar.count() > 0:
            sucesso_logs = gravar_delta(
                df=df_logs_para_gravar,
                camada="", # Vazio para salvar na raiz do Data Lake
                tabela=TABELA_DQ,
                storage_opts=STORAGE_OPTIONS,
                mode="append",
                particionar=False
            )
            if sucesso_logs:
                print("Logs de qualidade unificados salvos com sucesso!")
else:
    print("Nenhuma alteração física realizada no Data Lake.")

## Preparar referências externas para regras 7 e 8

In [0]:
print("===== VALIDAÇÃO FINAL =====")

if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_validacao_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    print(f"Registros totais na Silver {TABELA_ALVO}:", df_validacao_silver.count())
    display(df_validacao_silver.limit(20))

if delta_existe("silver", "dq_monitoring_logs", STORAGE_OPTIONS):
    df_logs_validacao = ler_delta("silver", "dq_monitoring_logs", STORAGE_OPTIONS) \
        .filter(F.col("tabela") == TABELA_ALVO)
    print(f"Total de violações registradas para {TABELA_ALVO}:", df_logs_validacao.count())
    display(df_logs_validacao.orderBy(F.col("timestamp_execucao").desc()).limit(20))

##  Validação Final

In [0]:
print("===== VALIDAÇÃO FINAL =====")

if delta_existe("silver", TABELA_ALVO, STORAGE_OPTIONS):
    df_validacao_silver = ler_delta("silver", TABELA_ALVO, STORAGE_OPTIONS)
    print(f"Registros totais na Silver {TABELA_ALVO}:", df_validacao_silver.count())
    display(df_validacao_silver.limit(20))

if delta_existe("silver", "dq_monitoring_logs", STORAGE_OPTIONS):
    df_logs_validacao = ler_delta("silver", "dq_monitoring_logs", STORAGE_OPTIONS) \
        .filter(F.col("tabela") == TABELA_ALVO)
    print(f"Total de violações registradas para {TABELA_ALVO}:", df_logs_validacao.count())
    display(df_logs_validacao.orderBy(F.col("timestamp_execucao").desc()).limit(20))